# Concurrency & Recovery Demos

Maps Neo4j to Elmasri & Navathe ch. 21–22. Runs against the 10-patient subset from `load_subset.py`.

**Prerequisites**
- `docker compose up -d`
- `python load_subset.py`

**Demo isolation.** Writes only touch demo-scoped properties (`p.demo_*`) on real PATIENT nodes; ephemeral entities carry `:DemoNode`. The setup cell wipes prior demo state, so the notebook is idempotent.

Each cell prints a slice of `query.log` or `debug.log` at the end — no terminal-tailing needed.

| # | Theme       | Demo                              |
|---|-------------|-----------------------------------|
| 1 | Transaction | Atomic multi-entity rollback      |
| 2 | Transaction | Checkpoint + WAL truncation       |
| 3 | Transaction | Crash recovery via WAL replay     |
| 4 | Concurrency | Non-repeatable read               |
| 5 | Concurrency | Lost update (naive vs atomic SET) |


In [1]:
import os, time, uuid, subprocess
from threading import Event, Thread, Barrier

import neo4j
from neo4j.exceptions import ConstraintError
from dotenv import load_dotenv

load_dotenv()
URI       = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USER      = os.getenv("NEO4J_USERNAME", "neo4j")
PWD       = os.getenv("NEO4J_PASSWORD", "password123")
DB        = os.getenv("NEO4J_DATABASE", "neo4j")
CONTAINER = "neo4j-demo"
RUN_ID    = uuid.uuid4().hex[:8]


def session():
    return driver.session(database=DB)


def show_log(logfile, n=20, grep=None):
    cmd = (f"grep -iE '{grep}' /logs/{logfile} 2>/dev/null | tail -n {n}"
           if grep else f"tail -n {n} /logs/{logfile}")
    out = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c", cmd],
        capture_output=True, text=True, timeout=15,
    ).stdout or "(no matches)"
    print(f"--- /logs/{logfile}  ({grep or 'tail'}) ---\n{out}")


def cleanup():
    with session() as s:
        s.run("MATCH (n:DemoNode) DETACH DELETE n").consume()
        s.run("MATCH (p:PATIENT) "
              "REMOVE p.demo_note, p.demo_last_visit, "
              "       p.demo_visit_count, p.demo_run_id").consume()


driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD))
driver.verify_connectivity()
cleanup()

with session() as s:
    row = s.run("""
        MATCH (p:PATIENT)-[:HAS_IMAGE]->(i:IMAGE)
        WHERE NOT p:DemoNode
        RETURN p.patient_id AS pid, i.instance_uid AS uid LIMIT 1
    """).single()
if not row:
    raise RuntimeError("No PATIENT/IMAGE found. Run `python load_subset.py`.")

PID          = row["pid"]
EXISTING_UID = row["uid"]
print(f"RUN_ID={RUN_ID}  PID={PID}")
print(f"sample instance_uid: {EXISTING_UID}")

RUN_ID=6e3cab29  PID=27
sample instance_uid: 1.3.6.1.4.1.9590.100.1.2.59620512812470186337816449881316634272


## Demo 1 — Atomic multi-entity rollback

ACID atomicity: a transaction is all-or-nothing.

One transaction does four steps:
1. update `PATIENT.demo_last_visit`
2. create an `:Annotation:DemoNode`
3. link it to the PATIENT and one of its real IMAGEs
4. **inject failure** — create an `:IMAGE` with an existing `instance_uid` (violates the `UNIQUE` constraint, raises `ConstraintError`, rolls back)

BEFORE and AFTER snapshots must match: no orphan node, no edge, no timestamp change.

In [2]:
def state():
    with session() as s:
        return dict(s.run("""
            MATCH (p:PATIENT {patient_id: $pid})
            OPTIONAL MATCH (a:Annotation:DemoNode)
            RETURN toString(p.demo_last_visit) AS last_visit,
                   count(a) AS annotations
        """, pid=PID).single())


with session() as s:
    s.run("MATCH (p:PATIENT {patient_id: $pid}) "
          "SET p.demo_last_visit = datetime('2025-01-01')",
          pid=PID).consume()

before = state()
print(f"BEFORE: {before}")

aid = f"{RUN_ID}-rb"
try:
    with session() as s, s.begin_transaction() as tx:
        tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
               "SET p.demo_last_visit = datetime('2026-05-14')", pid=PID)
        tx.run("CREATE (a:Annotation:DemoNode {annotation_id: $aid})", aid=aid)
        tx.run("""
            MATCH (a:Annotation {annotation_id: $aid}),
                  (p:PATIENT {patient_id: $pid})-[:HAS_IMAGE]->(i:IMAGE)
            WITH a, p, i LIMIT 1
            MERGE (a)-[:ANNOTATES]->(p)
            MERGE (a)-[:ABOUT_IMAGE]->(i)
        """, aid=aid, pid=PID)
        print("steps 1/2/3 staged")
        tx.run("CREATE (:IMAGE {instance_uid: $uid})", uid=EXISTING_UID)
except ConstraintError as e:
    print(f"ConstraintError -> auto-rollback: {e.message}")

after = state()
print(f"AFTER : {after}")
assert before == after, "Atomicity violated"
print("\nACID atomicity verified: failed transaction is a no-op.\n")

show_log("query.log", grep="Annotation|demo_last_visit")

BEFORE: {'last_visit': '2025-01-01T00:00:00Z', 'annotations': 0}
steps 1/2/3 staged
ConstraintError -> auto-rollback: Node(88) already exists with label `IMAGE` and property `instance_uid` = '1.3.6.1.4.1.9590.100.1.2.59620512812470186337816449881316634272'
AFTER : {'last_visit': '2025-01-01T00:00:00Z', 'annotations': 0}

ACID atomicity verified: failed transaction is a no-op.

--- /logs/query.log  (Annotation|demo_last_visit) ---
(no matches)


## Demo 2 — Checkpoint + WAL truncation

The WAL grows on every commit. A checkpoint flushes dirty pages, writes a checkpoint marker, and lets older WAL segments be pruned.

This bounds recovery time: Demo 3 only needs to replay from the *last checkpoint*, not from db creation.

The container is configured (`docker-compose.yml`) to checkpoint every **10 log chunks** (≈ 40 transactions in this workload) or every **5 seconds**. The cell counts `"Checkpoint started"` lines in `debug.log` before and after 200 writes — the **delta** is the demo's artifact.

*Note:* `db.checkpoint.interval.tx` in Neo4j 5 counts *log chunks*, not raw transactions (the trigger reason in the log says `"every N log chunks threshold"`). `CALL db.checkpoint()` is enterprise-only — we rely on the auto-trigger.

In [3]:
def n_checkpoints():
    p = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c",
         "grep -ci 'checkpoint started' /logs/debug.log || true"],
        capture_output=True, text=True,
    )
    return int(p.stdout.strip() or 0)


before = n_checkpoints()
print(f"BEFORE: {before} 'Checkpoint started' lines in debug.log")

N = 200
with session() as s:
    for i in range(N):
        s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_note = $v",
              pid=PID, v=f"tick-{i}").consume()
print(f"\n{N} writes done. Auto-checkpoint thresholds: 10 log chunks (~40 txs) "
      f"or every 5 s.")

time.sleep(6)  # let the time-based trigger fire at least once

after = n_checkpoints()
print(f"\nAFTER : {after} 'Checkpoint started' lines in debug.log")
print(f"DELTA : {after - before} checkpoint(s) fired during this cell.\n")

show_log("debug.log", grep="checkpoint started|prun")

BEFORE: 17 'Checkpoint started' lines in debug.log

200 writes done. Auto-checkpoint thresholds: 10 log chunks (~40 txs) or every 5 s.

AFTER : 19 'Checkpoint started' lines in debug.log
DELTA : 2 checkpoint(s) fired during this cell.

--- /logs/debug.log  (checkpoint started|prun) ---
2026-05-14 15:24:29.143+0000 INFO  [o.n.k.i.t.l.c.CheckPointerImpl] [neo4j/702299b5] Checkpoint triggered by "Recovery completed." @ txId: 755, append index: 755 checkpoint started...
2026-05-14 15:24:29.242+0000 INFO  [o.n.k.i.t.l.p.LogPruningImpl] [neo4j/702299b5] No log version pruned. The strategy used was '2 days 2147483648 size'. 
2026-05-14 15:24:39.457+0000 INFO  [o.n.k.i.t.l.c.CheckPointerImpl] [neo4j/702299b5] Checkpoint triggered by "Scheduled checkpoint for every 50 log chunks threshold" @ txId: 1044, append index: 1044 checkpoint started...
2026-05-14 15:24:39.567+0000 INFO  [o.n.k.i.t.l.p.LogPruningImpl] [neo4j/702299b5] No log version pruned. The strategy used was '2 days 2147483648 size'.

## Demo 3 — Crash recovery via WAL replay

Durability: a committed transaction survives any failure.

The cell commits a `:CrashMarker`, then `docker kill -s SIGKILL` the container, restarts it, prints recovery lines from `debug.log`, and queries for the marker — which must still exist.

This is the demo Aura cannot run.

In [4]:
marker = f"{RUN_ID}-CRASH-{int(time.time())}"

with session() as s:
    s.run("CREATE (:CrashMarker:DemoNode {marker_id: $m, written_at: datetime()})",
          m=marker).consume()
print(f"Committed marker {marker!r}")

driver.close()
subprocess.run(["docker", "kill", "-s", "SIGKILL", CONTAINER],
               check=True, capture_output=True)
print("SIGKILL sent")
subprocess.run(["docker", "start", CONTAINER],
               check=True, capture_output=True)
print("Container restarted; waiting for Bolt...")

for _ in range(90):
    try:
        driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD))
        driver.verify_connectivity()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Bolt didn't come back in 90s")
print("Bolt is back.\n")

show_log("debug.log", n=30, grep="recover|replay")

with session() as s:
    rec = s.run("MATCH (m:CrashMarker {marker_id: $m}) "
                "RETURN m.marker_id AS id, toString(m.written_at) AS t",
                m=marker).single()
print(f"\nMarker recovered: {rec}")
assert rec, "Durability violated"

Committed marker '6e3cab29-CRASH-1778772645'
SIGKILL sent
Container restarted; waiting for Bolt...
Bolt is back.

--- /logs/debug.log  (recover|replay) ---
2026-05-14 15:28:03.965+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  20% completed
2026-05-14 15:28:03.965+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  30% completed
2026-05-14 15:28:03.966+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  40% completed
2026-05-14 15:28:03.966+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  50% completed
2026-05-14 15:28:04.336+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  60% completed
2026-05-14 15:28:04.336+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  70% completed
2026-05-14 15:28:04.336+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  80% completed
2026-05-14 15:28:04.337+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5]  90% completed
2026-05-14 15:28:04.337+0000 INFO  [o.n.k.r.Recovery] [neo4j/702299b5] 100% completed
2026-05-14 15:28:04.689+0000 INFO  [o.n.k.d.Database] [neo4j/702299b5]

## Demo 4 — Non-repeatable read

Neo4j defaults to READ COMMITTED. That prevents *dirty* reads but allows *non-repeatable* ones: two reads of the same value in one transaction can differ if another transaction commits between them.

```
READER tx:  read #1 -------------- read #2 -- commit
WRITER tx:           SET -- commit
```

The two threads synchronise via `Event`s, so the ordering — and therefore the outcome — is deterministic.

In [5]:
with session() as s:
    s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_note = 'initial'",
          pid=PID).consume()

phase1, phase2 = Event(), Event()
reads = {}


def reader():
    with session() as s, s.begin_transaction() as tx:
        reads["r1"] = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                             "RETURN p.demo_note AS n", pid=PID).single()["n"]
        print(f"reader read#1 = {reads['r1']!r}")
        phase1.set()
        phase2.wait(timeout=10)
        reads["r2"] = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                             "RETURN p.demo_note AS n", pid=PID).single()["n"]
        print(f"reader read#2 = {reads['r2']!r}   (same tx, after writer commit)")


def writer():
    phase1.wait()
    with session() as s:
        s.run("MATCH (p:PATIENT {patient_id: $pid}) "
              "SET p.demo_note = 'mutated'", pid=PID).consume()
    print("writer committed")
    phase2.set()


a, b = Thread(target=reader), Thread(target=writer)
a.start(); b.start(); a.join(); b.join()

assert reads["r1"] != reads["r2"], "non-repeatable read did not occur"
print(f"\n✓ non-repeatable read: r1={reads['r1']!r}  r2={reads['r2']!r}\n")

show_log("query.log", grep="demo_note")

reader read#1 = 'initial'
writer committed
reader read#2 = 'mutated'   (same tx, after writer commit)

✓ non-repeatable read: r1='initial'  r2='mutated'

--- /logs/query.log  (demo_note) ---
(no matches)


## Demo 5 — Lost update: naive RMW vs atomic SET

When application code does its own read-modify-write across statements, the read takes no lock. Two threads can both read `v`, both write `v+1`. One increment vanishes silently.

Both threads use a `Barrier` to read in lock-step, so the race is **deterministic**:

| Round  | Pattern                          | Expected | Actual                              |
|--------|----------------------------------|----------|-------------------------------------|
| Naive  | `v = SELECT; SET = v+1`          | 200      | exactly **100** (every iteration both reads see the same `v`) |
| Atomic | `SET p.x = p.x + 1`              | 200      | exactly **200**                     |

Retries can't save you — lost updates raise no error.

In [6]:
N = 100


def naive_thread(barrier):
    with session() as s:
        for _ in range(N):
            with s.begin_transaction() as tx:
                v = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                           "RETURN coalesce(p.demo_visit_count, 0) AS v",
                           pid=PID).single()["v"]
                barrier.wait()
                tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                       "SET p.demo_visit_count = $v", pid=PID, v=v + 1)


def atomic_thread():
    with session() as s:
        for _ in range(N):
            s.run("MATCH (p:PATIENT {patient_id: $pid}) "
                  "SET p.demo_visit_count = coalesce(p.demo_visit_count, 0) + 1",
                  pid=PID).consume()


def reset_count():
    with session() as s:
        s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_visit_count = 0",
              pid=PID).consume()


def get_count():
    with session() as s:
        return s.run("MATCH (p:PATIENT {patient_id: $pid}) "
                     "RETURN p.demo_visit_count AS v", pid=PID).single()["v"]


def run_pair(target, *args):
    ts = [Thread(target=target, args=args) for _ in range(2)]
    for t in ts: t.start()
    for t in ts: t.join()


reset_count()
print(f"NAIVE (Barrier-locked), 2 threads x {N}:")
run_pair(naive_thread, Barrier(2))
nv = get_count()
print(f"  -> {nv}   (expected {N}, lost {2*N - nv})\n")
assert nv == N

reset_count()
print(f"ATOMIC SET, 2 threads x {N}:")
run_pair(atomic_thread)
av = get_count()
print(f"  -> {av}   (expected {2*N})\n")
assert av == 2 * N

show_log("query.log", grep="demo_visit_count")
cleanup()
print("\nDemo state cleaned.")

NAIVE (Barrier-locked), 2 threads x 100:
  -> 100   (expected 100, lost 100)

ATOMIC SET, 2 threads x 100:
  -> 200   (expected 200)

--- /logs/query.log  (demo_visit_count) ---
(no matches)

Demo state cleaned.
